# Exploring the TMTC gene family with NCBI and UniProt

This notebook demonstrates the reusable pipeline in `bio_database_explorer`. The unknown sequence was identified as **TMTC2** in the original NCBI BLAST run. We then collect MANE Select transcripts and reviewed human proteins for the TMTC family and calculate sequence features.

## 1. Load and inspect the query locally

This step does not use the network.

In [ ]:
from pathlib import Path

fasta_path = Path("../data/mysterious_sequence.fasta")
query_sequence = "".join(
    line.strip()
    for line in fasta_path.read_text(encoding="utf-8").splitlines()
    if not line.startswith(">")
)

print(f"Query length: {len(query_sequence)} nt")
print(f"First 60 nt: {query_sequence[:60]}")

In [ ]:
from bio_database_explorer import gc_content

print(f"Query GC fraction: {gc_content(query_sequence):.3f}")

## 2. Optional remote BLAST

The following call submits the query to NCBI and may take several minutes. It is intentionally commented out so opening the notebook does not create a network request.

In [ ]:
from bio_database_explorer.blast import identify_sequence

RUN_REMOTE_BLAST = False
blast_hits = identify_sequence(query_sequence, max_hits=5) if RUN_REMOTE_BLAST else []
[(hit.accession, hit.identity_percent, hit.title) for hit in blast_hits]

## 3. Build the TMTC family table

NCBI requires a contact email. Set `NCBI_EMAIL` in the environment before running this cell. The output is generated from live databases and can change as records are updated.

In [ ]:
import os

from bio_database_explorer import explore_gene_family

email = os.getenv("NCBI_EMAIL")
if not email:
    raise RuntimeError("Set the NCBI_EMAIL environment variable before this step")

family_table = explore_gene_family(
    family="TMTC",
    email=email,
    api_key=os.getenv("NCBI_API_KEY"),
    motif="DW",
)

family_table.drop(columns=["nucleotide_sequence", "protein_sequence"])

## 4. Export reproducible results

CSV is used as the default portable format. The generated file is ignored by Git because it can be rebuilt from NCBI and UniProt.

In [ ]:
output_path = Path("../results/tmtc_family.csv")
output_path.parent.mkdir(exist_ok=True)
family_table.to_csv(output_path, index=False)
output_path

## Interpretation

- BLAST similarity supports TMTC2 as the source of the query sequence.
- The family workflow makes the analysis extensible: another prefix can be supplied without rewriting the data-access code.
- GC fraction and motif presence are descriptive features. Their biological meaning requires additional domain analysis and literature review.